# 01 — Esplorazione del Dataset e Preparazione Manifest Binario

## Scopo del Notebook
1. Caricare e ispezionare i file ufficiali (`eccdna_disease_detection_metadata.tsv` e `eccdna_disease_detection.body.fa`).
2. Analizzare la distribuzione del **target binario** (`disease_binary_label`: `0` = Stabile/Healthy, `1` = Hotspot/Disease-associated), la lunghezza delle sequenze, il contenuto GC e la frazione di basi ambigue (`N`).
3. Verificare l'integrità dello split genomico `split_cluster` (`train`, `val`, `test`) garantendo **zero data leakage** tra cluster.
4. Calcolare i limiti teorici di occupazione e sparsità dei $k$-mer per $k \in [4, 10]$.
5. Esportare il **Manifest Binario Bilanciato** (`data/processed/siamese_primary_manifest.tsv`) pronto per la generazione FCGR e la Rete Siamese.

In [ ]:
# Su Google Colab esegui prima il mount del Drive (se non già montato):
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
import sys

# Rilevamento automatico Google Colab vs Ambiente Locale
COLAB_ROOT = Path('/content/drive/MyDrive/eccdna_project')
if COLAB_ROOT.exists():
    PROJECT_ROOT = COLAB_ROOT
else:
    CURRENT_DIR = Path.cwd().resolve()
    PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == 'notebooks' else CURRENT_DIR

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
RESULTS_DIR = PROJECT_ROOT / 'results' / 'fcgr_analysis'
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

META_PATH = RAW_DIR / 'eccdna_disease_detection_metadata.tsv'
FASTA_PATH = RAW_DIR / 'eccdna_disease_detection.body.fa'
MANIFEST_PATH = PROCESSED_DIR / 'siamese_primary_manifest.tsv'

print('PROJECT_ROOT :', PROJECT_ROOT)
print('META_PATH    :', META_PATH, '-> exists:', META_PATH.exists())
print('FASTA_PATH   :', FASTA_PATH, '-> exists:', FASTA_PATH.exists())

## 1. Caricamento del Metadata TSV e Controllo Qualità
Ispezioniamo le colonne, i valori mancanti e la distribuzione del target binario `disease_binary_label` (`0` vs `1`). Le colonne cliniche (`disease`, `tissue`, `cell_line`, `source_db`) vengono mantenute come metadati accessori.

In [ ]:
metadata = pd.read_csv(META_PATH, sep='\t', low_memory=False)

print(f'Numero totale record : {len(metadata):,}')
print(f'Numero colonne       : {metadata.shape[1]}')
print(f'ID unici             : {metadata["id"].nunique():,}')
print(f'ID duplicati         : {metadata["id"].duplicated().sum():,}')

display(metadata[['id', 'disease_binary_label', 'disease_binary_name', 'disease', 'chrom', 'start', 'end', 'length', 'gc', 'n_fraction', 'source_db', 'split_cluster']].head())

In [ ]:
missing_values = {'', 'NA', 'NaN', 'nan', 'None', 'none', 'NULL', 'null'}
cols_to_check = ['id', 'disease_binary_label', 'disease', 'tissue', 'cell_line', 'source_db', 'split_cluster']

print('=== CONTROLLO VALORI MANCANTI ===')
for col in cols_to_check:
    missing = metadata[col].fillna('').astype(str).str.strip().isin(missing_values).sum()
    pct = missing / len(metadata) * 100.0
    print(f'{col:22s}: {missing:10,d} missing ({pct:6.2f}%)')

## 2. Distribuzione del Target Binario e Verifica Split (`split_cluster`)
Verifichiamo il bilanciamento tra classe `0` e classe `1` nei tre split (`train`, `val`, `test`) e controlliamo che nessun `cluster_id` sia condiviso tra split diversi.

In [ ]:
binary_counts = metadata['disease_binary_label'].value_counts().sort_index()
binary_pct = metadata['disease_binary_label'].value_counts(normalize=True).sort_index().mul(100).round(2)

summary_target = pd.DataFrame({'Count': binary_counts, 'Percentage (%)': binary_pct})
summary_target.index = ['0 (Stable / Healthy)', '1 (Hotspot / Disease-associated)']
display(summary_target)

split_table = pd.crosstab(metadata['split_cluster'], metadata['disease_binary_label'], margins=True)
print('\n=== DISTRIBUZIONE PER SPLIT_CLUSTER ===')
display(split_table)

# Verifica assenza di data leakage su cluster_id
leaky_clusters = (metadata.groupby('cluster_id')['split_cluster'].nunique() > 1).sum()
print(f'Cluster presenti in più di uno split (Data Leakage check): {leaky_clusters}')
assert leaky_clusters == 0, 'Attenzione: rilevato leakage tra split!'

## 3. Caratteristiche Genomiche (Lunghezza, GC Content, Limite Teorico FCGR)
Analizziamo la lunghezza delle sequenze e calcoliamo l'occupazione massima teorica di una matrice FCGR ($4^k$ celle) al variare di $k \in [4, 10]$.

In [ ]:
print('=== STATISTICHE LUNGHEZZA (bp) E GC CONTENT ===')
display(metadata.groupby('disease_binary_label')[['length', 'gc', 'n_fraction']].describe().round(3))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].hist(np.log10(metadata['length'].clip(lower=1)), bins=80, color='#2B6CB0', alpha=0.85)
axes[0].set_xlabel('log10(Lunghezza sequenza in bp)')
axes[0].set_ylabel('Numero di sequenze')
axes[0].set_title('Distribuzione log10(Lunghezza) eccDNA')
axes[0].grid(alpha=0.3)

axes[1].hist(metadata['gc'], bins=60, color='#2F855A', alpha=0.85)
axes[1].set_xlabel('GC Content')
axes[1].set_ylabel('Numero di sequenze')
axes[1].set_title('Distribuzione GC Content')
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Analisi teorica: quanti k-mer osservabili vs 4^k celle della FCGR
kmer_stats = []
median_len = float(metadata['length'].median())

for k in range(4, 11):
    possible_kmers = 4 ** k
    fcgr_side = 2 ** k
    median_obs_kmers = max(0.0, median_len - k + 1)
    max_occ_pct = min(median_obs_kmers, possible_kmers) / possible_kmers * 100.0
    min_sparsity_pct = 100.0 - max_occ_pct
    kmer_stats.append({
        'k': k,
        'fcgr_shape': f'{fcgr_side}x{fcgr_side}',
        'total_cells (4^k)': possible_kmers,
        'median_seq_kmers': int(median_obs_kmers),
        'max_theoretical_occupancy_%': round(max_occ_pct, 3),
        'min_theoretical_sparsity_%': round(min_sparsity_pct, 3),
    })

kmer_stats_df = pd.DataFrame(kmer_stats)
print(f'Lunghezza mediana globale del dataset: {median_len:.0f} bp')
display(kmer_stats_df)

## 4. Verifica Allineamento tra Metadata TSV e File FASTA
Controlliamo che i primi record del file FASTA corrispondano esattamente agli ID e alle lunghezze registrate nel TSV.

In [ ]:
from src.fcgr import load_fasta_sequences

sample_check_ids = metadata['id'].iloc[:10].astype(str).tolist()
sample_seqs = load_fasta_sequences(FASTA_PATH, sample_check_ids)

print(f'ID richiesti: {len(sample_check_ids)} | ID trovati nel FASTA: {len(sample_seqs)}')
for sid in sample_check_ids[:3]:
    row = metadata.loc[metadata['id'] == sid].iloc[0]
    s = sample_seqs[sid]
    print(f'ID: {sid} | Label binaria: {row["disease_binary_label"]} | Len TSV: {row["length"]} | Len FASTA: {len(s)} | Match: {row["length"] == len(s)}')

## 5. Costruzione ed Esportazione del Manifest Binario Bilanciato
Costruiamo il manifest per la pipeline FCGR + Rete Siamese (`siamese_primary_manifest.tsv`):
- `class_id` = `disease_binary_label` (`0` vs `1`).
- Per rendere il training e il benchmarking agili su Colab e perfettamente bilanciati, campioniamo in modo stratificato fino a `MAX_TRAIN_PER_CLASS` sequenze per ciascuna delle 2 classi nel training set (es. `25.000` per classe = `50.000` sequenze di train totali), mantenendo intatta la separazione di `split_cluster`.

In [ ]:
RANDOM_STATE = 42
MAX_TRAIN_PER_CLASS = 25_000  # 25k classe 0 + 25k classe 1 = 50,000 sequenze di train bilanciate

metadata['class_id'] = metadata['disease_binary_label'].astype(int)
metadata['disease_clean'] = np.where(metadata['class_id'] == 1, 'hotspot_disease', 'stable_healthy')
metadata['disease_group'] = metadata['disease_clean']

def assign_length_bin(length):
    if length < 300:
        return 'short'
    elif length <= 1000:
        return 'medium'
    return 'long'

metadata['length_bin'] = metadata['length'].apply(assign_length_bin)

# Campionamento bilanciato del Training Set
train_parts = []
train_full = metadata[metadata['split_cluster'] == 'train']
for cid in [0, 1]:
    sub = train_full[train_full['class_id'] == cid]
    n_take = min(len(sub), MAX_TRAIN_PER_CLASS)
    train_parts.append(sub.sample(n=n_take, replace=False, random_state=RANDOM_STATE + cid))

train_manifest = pd.concat(train_parts, ignore_index=True)
val_manifest = metadata[metadata['split_cluster'] == 'val'].copy()
test_manifest = metadata[metadata['split_cluster'] == 'test'].copy()

MANIFEST_COLUMNS = [
    'id', 'class_id', 'disease_binary_label', 'disease_clean', 'disease',
    'split_cluster', 'cluster_id', 'length', 'length_bin', 'gc', 'n_fraction',
    'source_db', 'tissue', 'chrom', 'start', 'end'
]
available_cols = [c for c in MANIFEST_COLUMNS if c in metadata.columns]

siamese_manifest = pd.concat([train_manifest, val_manifest, test_manifest], ignore_index=True)[available_cols].copy()
siamese_manifest.to_csv(MANIFEST_PATH, sep='\t', index=False)

CLASS_MAPPING = {'stable_healthy': 0, 'hotspot_disease': 1}
with open(PROCESSED_DIR / 'class_mapping.json', 'w', encoding='utf-8') as f:
    json.dump(CLASS_MAPPING, f, indent=4)

print('Manifest binario salvato in:', MANIFEST_PATH)
print('Shape totale manifest      :', siamese_manifest.shape)
display(pd.crosstab(siamese_manifest['disease_clean'], siamese_manifest['split_cluster']))